# EXPERIMENT REPORT: Implementing Retinal Latent Constraint (RLC) in Video Diffusion

---

## 1. The Initial Hypothesis: "Retinal Persistence"

**The Goal:** Video diffusion models (like AnimateDiff) often suffer from temporal jitter (flickering details) because each frame's noise is predicted somewhat independently. We hypothesized that we could enforce temporal stability mathematically by mimicking the human retina's persistence of vision.

**The Strategy:** We injected a custom mathematical constraint into the generation loop between the UNet's noise prediction and the Scheduler's step.

We built `apply_retinal_constraint` based on two pillars:

- **Trajectory Smoothing (Inertia):** We took the noisy image representations (latents) and forced them onto a smoothed timeline. If Frame 1 is at Point A, and Frame 3 is at Point C, Frame 2 must be pushed toward Point B.

- **Orthogonal Projection:** We split the model's noise prediction into "Motion" (parallel to the trajectory) and "Jitter" (perpendicular). We wanted to keep the motion and delete the jitter.

---


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import math
import cv2
import os
from PIL import Image
from diffusers import MotionAdapter, AnimateDiffPipeline, DDIMScheduler
from diffusers.utils import export_to_gif
from skimage.metrics import structural_similarity as ssim

os.makedirs('results', exist_ok=True)

print("Loading Model...")
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
model_id = "emilianJR/epiCRealism"
pipe = AnimateDiffPipeline.from_pretrained(model_id, motion_adapter=adapter, torch_dtype=torch.float16)
pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
pipe.scheduler = DDIMScheduler.from_config(
    pipe.scheduler.config,
    beta_schedule="linear",
    clip_sample=False,
    timestep_spacing="linspace",
    steps_offset=1
)
print("Model Loaded!")

def tensor_to_pil(video_tensor):
    if video_tensor.ndim == 5: video_tensor = video_tensor[0]
    video_tensor = video_tensor.permute(1, 2, 3, 0).cpu()
    video_np = (video_tensor.float().numpy() * 255).clip(0, 255).astype(np.uint8)
    return [Image.fromarray(f) for f in video_np]

@torch.no_grad()
def generate(prompt, num_frames=16, steps=20, rlc_fn=None, seed=42):
    generator = torch.Generator(device="cuda").manual_seed(seed)
    neg_prompt = "static, motionless, blurry, distorted, noise, grain, low quality, watermark, text"
    prompt_embeds, negative_embeds = pipe.encode_prompt(
        prompt, device="cuda", num_images_per_prompt=1, do_classifier_free_guidance=True,
        negative_prompt=neg_prompt
    )
    batch_embeds = torch.cat([negative_embeds, prompt_embeds]).repeat_interleave(num_frames, dim=0)
    latents = torch.randn((1, 4, num_frames, 64, 64), device="cuda", dtype=torch.float16, generator=generator)
    pipe.scheduler.set_timesteps(steps)
    
    for i, t in enumerate(pipe.scheduler.timesteps):
        latent_input = torch.cat([latents] * 2)
        latent_input = pipe.scheduler.scale_model_input(latent_input, t)
        noise_pred = pipe.unet(latent_input, t, encoder_hidden_states=batch_embeds).sample
        noise_uncond, noise_text = noise_pred.chunk(2)
        noise_pred = noise_uncond + 7.5 * (noise_text - noise_uncond)
        
        if rlc_fn is not None:
            noise_pred = rlc_fn(noise_pred=noise_pred, latents=latents, t=t, step_idx=i, total_steps=steps)
            
        latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample
        
    latents = 1 / 0.18215 * latents
    latents_2d = latents.permute(0, 2, 1, 3, 4).flatten(0, 1)
    video = pipe.vae.decode(latents_2d).sample
    video = video.reshape(1, num_frames, 3, 512, 512)
    video = video.permute(0, 2, 1, 3, 4)
    return video
    
PROMPT = "drone camera panning right, a tiger running fast through jungle, motion blur, 4k"


Loading Model...


c:\Anaconda\envs\VidDiff\lib\site-packages\huggingface_hub\utils\_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
The config attributes {'motion_activation_fn': 'geglu', 'motion_attention_bias': False, 'motion_cross_attention_dim': None} were passed to MotionAdapter, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: C:\Users\PAWAN SURYA\.cache\huggingface\hub\models--emilianJR--epiCRealism\snapshots\6522cf856b8c8e14638a0aaa7bd89b1b098aed17\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded!


c:\Anaconda\envs\VidDiff\lib\site-packages\diffusers\pipelines\pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `AnimateDiffPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(
c:\Anaconda\envs\VidDiff\lib\site-packages\diffusers\pipelines\pipeline_utils.py:2213: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `AnimateDiffPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


## 2. Timeline of Execution and Failures

### Phase 1: The "Deep Fried" Era
**The Code:** We wrote a function that calculated `trajectory_correction = latents_smooth - latents`. We directly subtracted this correction from the model's `noise_pred`.

**The Result:** The videos exploded into high-contrast colors, random static, and "deep-fried" artifacts.

**The Diagnosis (Unit Mismatch):** We made a physics error. `latents` represent **Position** (where the pixels are). `noise_pred` represents **Velocity** (where the pixels should go). By subtracting a position error directly from a velocity vector, the numbers became astronomically large. It was the equivalent of realizing your car is 5 meters off-center and instantly setting your steering wheel to "5 meters" instead of turning it 5 degrees.

---


In [2]:
def apply_phase1_rlc(noise_pred, latents, **kwargs):
    # Trajectory Smoothing: Average neighboring frames
    latents_padded = F.pad(latents, (0,0, 0,0, 1,1), mode="replicate")
    latents_smooth = (latents_padded[:, :, :-2] + latents_padded[:, :, 1:-1] + latents_padded[:, :, 2:]) / 3.0
    
    # Unit Mismatch: Direct Subtraction (The fatal error)
    trajectory_correction = latents_smooth - latents
    final_noise = noise_pred - trajectory_correction
    return final_noise

print("Running Phase 1: Deep Fried Era...")
v1 = generate(PROMPT, rlc_fn=apply_phase1_rlc)
export_to_gif(tensor_to_pil(v1), "results/phase1_deep_fried.gif")
print("Saved to results/phase1_deep_fried.gif")


Running Phase 1: Deep Fried Era...
Saved to results/phase1_deep_fried.gif


### Phase 2: The "Frozen World" Era
**The Code:** We implemented **Safe Normalization**. We updated the code to calculate the standard deviation of both the noise and the error. We scaled the correction so it never exceeded ~15% of the noise prediction's magnitude. We also added a Velocity Gate so static backgrounds wouldn't cause division-by-zero errors.

**The Result:** The mathematical explosions stopped. The videos were clean, but absolutely nothing moved.

**The Diagnosis (Averaging = Braking):** Our smoothing logic calculated `(Frame 1 + Frame 2 + Frame 3) / 3`. However, motion *is* deviation. When a tiger's leg moves forward, it mathematically deviates from the previous frame. By forcing frames to look like the average of their neighbors, we mathematically punished deviation, effectively deleting the motion.

---


In [3]:
def apply_phase2_rlc(noise_pred, latents, **kwargs):
    latents_padded = F.pad(latents, (0,0, 0,0, 1,1), mode="replicate")
    latents_smooth = (latents_padded[:, :, :-2] + latents_padded[:, :, 1:-1] + latents_padded[:, :, 2:]) / 3.0
    error_vector = latents_smooth - latents
    
    # Safe Normalization
    noise_mag = noise_pred.std()
    error_mag = error_vector.std()
    
    # Scale correction so it never exceeds 15% of noise magnitude
    scale = min(1, (noise_mag / error_mag) *0.15) if error_mag > 1e-6 else 0
    scaled_correction = error_vector * scale
    
    final_noise = noise_pred - scaled_correction
    return final_noise

print("Running Phase 2: Frozen World Era...")
v2 = generate(PROMPT, rlc_fn=apply_phase2_rlc)
export_to_gif(tensor_to_pil(v2), "results/phase2_frozen_world.gif")
print("Saved to results/phase2_frozen_world.gif")


Running Phase 2: Frozen World Era...
Saved to results/phase2_frozen_world.gif


### Phase 3: The Debugging & "Muddy Trash" Era
**The Code:** To figure out why the RLC was still generating "trash" artifacts on certain settings while the Baseline (no RLC) was completely static, we wrote `apply_retinal_constraint_debug` to print out the exact forces being applied at every timestep (`t=999` to `t=0`).

**The Result:** The logs proved our normalization math was perfect (`Ratio: 0.20`), but the visual output was still terrible.

**The Diagnosis (The Breakthrough):** We discovered two fundamental structural flaws in our approach:

1. **The Noise Trap (Why RLC was trash):** At step `t=999`, the "image" is just pure Gaussian noise. Our algorithm was smoothing random noise, which just creates "blurry noise." By forcing the model to adhere to this smoothed noise, we were instructing the UNet to sculpt the image out of static. We were constraining the foundation before the house was built.

2. **Frame Starvation (Why Baseline was static):** To save GPU memory, we had set `num_frames = 8`. However, the AnimateDiff v1.5 adapter is explicitly trained on 16-frame clips. When we fed it 8 frames, its internal time-awareness broke. It squashed the timeline, defaulting to a static image because it didn't think it had enough "time" to execute a motion.

---


In [4]:
def apply_phase3_debug_rlc(noise_pred, latents, t, **kwargs):
    latents_padded = F.pad(latents, (0,0, 0,0, 1,1), mode="replicate")
    latents_smooth = (latents_padded[:, :, :-2] + latents_padded[:, :, 1:-1] + latents_padded[:, :, 2:]) / 3.0
    error_vector = latents - latents_smooth  # Changed sign for logging consistency
    
    noise_mag = noise_pred.std()
    error_mag = error_vector.std()
    scale = (noise_mag / error_mag) * 0.15 if error_mag > 1e-6 else 0
    scaled_correction = error_vector * scale
    final_noise = noise_pred - scaled_correction
    
    adj_mag = scaled_correction.std()
    ratio = adj_mag / (noise_mag + 1e-6)
    
    # Log to prove normalization math is perfect
    if kwargs.get('step_idx') % 5 == 0:
        print(f"   [t={t.item():03d}] Noise: {noise_mag:.4f} | Error: {error_mag:.4f} | Adj: {adj_mag:.4f} | Ratio: {ratio:.2f}")
    
    return final_noise

print("Running Phase 3: Debugging & Muddy Trash Era (8 frames)...")
# Note: num_frames=8 triggers the "Frame Starvation" pathology
v3 = generate(PROMPT, num_frames=8, rlc_fn=apply_phase3_debug_rlc)
export_to_gif(tensor_to_pil(v3), "results/phase3_trash.gif")
print("Saved to results/phase3_trash.gif")


Running Phase 3: Debugging & Muddy Trash Era (8 frames)...
   [t=999] Noise: 1.0068 | Error: 0.7446 | Adj: 0.1511 | Ratio: 0.15
   [t=736] Noise: 1.1885 | Error: 1.0010 | Adj: 0.1783 | Ratio: 0.15
   [t=473] Noise: 1.1768 | Error: 1.2471 | Adj: 0.1765 | Ratio: 0.15
   [t=210] Noise: 1.0400 | Error: 1.4668 | Adj: 0.1560 | Ratio: 0.15
Saved to results/phase3_trash.gif


## 3. The Pivot & New Architecture: "Gradient Consensus"
**Conclusion from Phase 3:** Our core hypothesis — constraining the Latents (the image pixels) — was structurally flawed because it directly fought the diffusion denoising process.

**The New Strategy:** Instead of forcing the image pixels to average out, we decided to smooth the *Instructions* the model was giving.

We threw away the complex Trajectory/Orthogonal math and implemented a much simpler, safer approach:

1. **Upgraded to 16 Frames:** We fixed the static baseline by feeding the model the `num_frames=16` it required, using VAE Slicing and Tiling to ensure it fit into an 8GB VRAM GPU.

2. **`apply_gradient_smoothing`:** We wrote a new function that applies a temporal blur `[1, 2, 1]` exclusively to the `noise_pred` (the gradient update), not the latents.

**How it works:** If Frame 1 wants to color a pixel Red, and Frame 2 wants to color it Blue, Gradient Consensus forces them to agree on Purple for that specific timestep's update. This mathematically prevents flickering (inconsistent updates) while freely allowing the image to move across the screen. We also applied a schedule to shut off the constraint in the final 20% of steps so fine textures (like fur) could resolve without being blurred.

---


## 4. Quantitative Evaluation of Gradient Consensus
### Phase 4: The "Contrasting Colors" Era

**The Code:** We deployed `apply_gradient_smoothing` with a `[1, 2, 1]` kernel at `strength=0.3`, active for the first 80% of timesteps. We also added **Variance Normalization** — after smoothing, we rescaled the output so the standard deviation matched the original `noise_pred`, to prevent the mathematical averaging from shrinking the signal magnitude (which was causing hyper-saturated, high-contrast artifacts).

**The Result:** The deep-fried explosions stopped, but visual inspection was ambiguous — the tiger was visible and moving, but the colors still looked unnatural. We could no longer trust our eyes.

**The Fix — Objective Metrics:** We introduced two quantitative metrics to replace subjective visual inspection:

- **SSIM (Structural Similarity Index):** Measures how structurally similar Frame N is to Frame N+1. Higher = more temporally stable.
- **PTD (Pixel-wise Temporal Difference):** Measures the average absolute pixel change between consecutive frames. Lower = less random flickering.

**The Multi-Prompt Evaluation:** We ran the algorithm across **4 different prompts** (tiger in jungle, dog on beach, car on highway, person in snow) to ensure the results weren't prompt-specific.

**The Numbers:**

| Prompt | Base SSIM | RLC SSIM | Base PTD | RLC PTD | Winner |
|---|---|---|---|---|---|
| Tiger running through jungle | **0.6133** | 0.4272 | **12.90** | 22.63 | BASE |
| Golden retriever on beach | **0.6586** | 0.4505 | **10.74** | 17.49 | BASE |
| Car driving highway sunset | **0.8313** | 0.7656 | **6.32** | 8.50 | BASE |
| Person walking snowy forest | **0.7454** | 0.5756 | **9.25** | 11.98 | BASE |

**The Diagnosis (Gradient Contamination):** The baseline won on **every prompt, on both metrics**. The RLC made videos *less* temporally stable, not more. The root cause: by blending Frame 2's noise prediction with Frame 1's and Frame 3's predictions at `strength=0.3`, we were **contaminating** each frame's specific update instructions with instructions meant for neighboring frames. The model was receiving a *compromise* of three different sets of directions — producing frames that didn't match what *any* of the three predictions intended.

---


In [5]:
def apply_phase4_gradient_rlc(noise_pred, step_idx, total_steps, **kwargs):
    progress = step_idx / total_steps
    
    # Shut off constraint in the final 20% of steps
    if progress < 0.8:
        original_std = noise_pred.std()
        n_padded = F.pad(noise_pred, (0,0, 0,0, 1,1), mode="replicate")
        n_smooth = (n_padded[:,:,:-2] + 2*n_padded[:,:,1:-1] + n_padded[:,:,2:]) / 4.0
        
        final_noise = noise_pred + 0.3 * (n_smooth - noise_pred)
        new_std = final_noise.std()
        
        if new_std > 1e-6:
            final_noise = final_noise * (original_std / new_std)
        return final_noise
        
    return noise_pred

print("Running Phase 4: Gradient Consensus Era...")
v4 = generate(PROMPT, rlc_fn=apply_phase4_gradient_rlc)
export_to_gif(tensor_to_pil(v4), "results/phase4_gradient_consensus.gif")
print("Saved to results/phase4_gradient_consensus.gif")


Running Phase 4: Gradient Consensus Era...
Saved to results/phase4_gradient_consensus.gif


## 5. Phase 5: Anomaly-Based Selective Smoothing
### The Hypothesis Revision

Since blind smoothing was destructive, we hypothesized that a **targeted** approach could work: instead of smoothing every pixel's prediction, detect which spatial locations have anomalously inconsistent predictions across adjacent frames (actual flickering spots) and only smooth those. Leave everything else completely untouched.

### The Implementation

`apply_anomaly_rlc` was built with three safeguards:

1. **Anomaly Detection:** Compute the temporal difference of `noise_pred` between adjacent frames. Only flag locations where the difference exceeds **2× the median** — these are the statistical outliers (flickering pixels).
2. **Strict Schedule:** Only active between 40% and 90% of the diffusion process (skip noise phase + detail phase).
3. **Conservative Strength:** Maximum `0.1` (vs. the previous `0.3`) with a **cosine decay** within the active window.

### The Multi-Prompt Evaluation

| Prompt | Base SSIM | RLC SSIM | Base PTD | RLC PTD | Winner |
|---|---|---|---|---|---|
| Tiger running through jungle | 0.6133 | 0.6112 | 12.90 | 12.97 | BASE |
| Golden retriever on beach | 0.6586 | 0.6553 | 10.74 | 10.84 | BASE |
| Car driving highway sunset | 0.8313 | 0.8302 | 6.32 | 6.34 | BASE |
| Person walking snowy forest | 0.7454 | 0.7416 | 9.25 | 9.36 | BASE |

**The Result:** The algorithm was no longer destructive — differences were less than 0.5% — but it was also completely **inert**. The baseline still won on every metric by a marginal amount.

**The Diagnosis (Redundancy):** AnimateDiff already contains purpose-built **temporal attention layers** (motion modules) that were specifically *trained* to enforce frame-to-frame consistency. Our external smoothing was either:
- **Too aggressive** → overrode the model's learned temporal behavior → destroyed the video
- **Too conservative** → the model's own temporal attention already handled consistency → our intervention was redundant

There is no "sweet spot" because the model's internal modules already occupy the correct operating point.

---


frame by frame comparision ssim
mse ssim map check if we can 

In [6]:
def apply_phase5_anomaly_rlc(noise_pred, step_idx, total_steps, **kwargs):
    progress = step_idx / total_steps
    out_dtype = noise_pred.dtype
    
    if progress < 0.4 or progress > 0.9:
        return noise_pred
        
    window_progress = (progress - 0.4) / 0.5
    strength = 0.1 * (0.5 * (1 + math.cos(math.pi * window_progress)))
    
    if strength < 1e-6: return noise_pred
    
    temporal_diff = torch.abs(noise_pred[:, :, 1:] - noise_pred[:, :, :-1])
    diff_magnitude = temporal_diff.mean(dim=1, keepdim=True)
    median_diff = diff_magnitude.median()
    threshold = 2.0 * median_diff
    
    anomaly_mask_partial = (diff_magnitude > threshold).float()
    anomaly_mask = torch.cat([anomaly_mask_partial[:, :, :1], anomaly_mask_partial], dim=2)
    
    n_padded = F.pad(noise_pred, (0,0, 0,0, 1,1), mode="replicate")
    n_smooth = (n_padded[:,:,:-2] + 2*n_padded[:,:,1:-1] + n_padded[:,:,2:]) / 4.0
    
    blend_mask = anomaly_mask * strength
    final_noise = noise_pred + blend_mask * (n_smooth - noise_pred)
    
    return final_noise.to(dtype=out_dtype)

print("Running Phase 5: Anomaly-Based Selective Smoothing...")
v5 = generate(PROMPT, rlc_fn=apply_phase5_anomaly_rlc)
export_to_gif(tensor_to_pil(v5), "results/phase5_anomaly.gif")
print("Saved to results/phase5_anomaly.gif")


Running Phase 5: Anomaly-Based Selective Smoothing...
Saved to results/phase5_anomaly.gif


## 6. Final Conclusions
### Conclusion 1: The RLC Hypothesis Is Structurally Incompatible with Diffusion Models

Every variant of external temporal constraint — whether applied to **Latents** (Phases 1–3) or to **Noise Predictions** (Phases 4–5) — either degraded or had zero effect on video quality. The fundamental issue is that diffusion models learn a precise mapping from noise to image across timesteps. Any external mathematical intervention disrupts this learned mapping. The model's internal temporal attention modules are already optimized for this task.

### Conclusion 2: The "Sweet Spot" Does Not Exist

We systematically swept the parameter space:

| Strength | Target | Schedule | Result |
|---|---|---|---|
| 0.15–0.20 | Latents | Full | Deep fried / Frozen |
| 0.15 | Latents (Orthogonal) | t=200–800 only | Muddy trash |
| 0.30 | Noise Pred (Global) | First 80% | Worse than baseline (all metrics) |
| 0.10 | Noise Pred (Anomaly-only) | 40–90% + Cosine | Identical to baseline |

### Conclusion 3: Proven Alternatives Exist

Techniques that work *with* the model's architecture rather than against it have been shown in literature to genuinely improve temporal consistency:

- **FreeInit** (built into `diffusers`): Re-initializes latent noise using low-frequency information from a first pass, giving the model a "warm start" on temporal structure without modifying any noise predictions.
- **Post-generation stabilization:** Optical flow-based video stabilization or frame interpolation (RIFE/FILM) applied after the video is fully generated.
- **Architecture-level improvements:** Newer models (AnimateDiff v3, SVD, CogVideoX) use improved temporal attention that natively reduces flickering.

### Conclusion 4: What We Learned

The experiment, while producing a negative result for the RLC hypothesis, provided valuable insights:

1. **Unit awareness matters:** Latent-space values and noise predictions operate on fundamentally different scales — mixing them without normalization causes catastrophic failure.
2. **Averaging kills motion:** Temporal smoothing and motion are mathematically opposing forces. Smoothing enforces similarity; motion requires difference.
3. **Quantitative metrics are essential:** Visual inspection of AI-generated video is unreliable. SSIM and PTD provided objective proof that contradicted visual intuition.
4. **Respect the model's learned behavior:** Pre-trained diffusion models have internally learned temporal dynamics. External constraints that conflict with this learned behavior are counterproductive.
